# 🚀 Tencent Hunyuan3D-2.1 Free GPU Backend for Web Blender Studio
### 🌟 Máy Chủ GPU Miễn Phí 100% (NVIDIA T4 16GB VRAM) Phục Vụ Tạo Khối 3D & PBR Texture Baking

Notebook này cung cấp backend AI 3D hoàn chỉnh cho **Web Blender Studio**:
- **Tạo Khối Hình Học Đa Hướng (3D DiT Flow Matching)**: Tự suy luận mặt sau và hai bên tự nhiên, không bị lỗi "trước sau như một".
- **PBR Texture Baking (Hunyuan3D-Paint)**: Tô vẽ vật liệu đa góc nhìn, tạo vân nổi, độ nhám và màu sắc thực tế (Full PBR Textured GLB).
- **Không Giới Hạn Quota ZeroGPU**: Không bị ngắt quãng sau 60 giây như trên HuggingFace Spaces công cộng.
- **Đường Dẫn Công Khai (Gradio Live Tunnel)**: Kết nối trực tiếp vào giao diện Web Blender Studio chỉ qua 1 đường link `https://xxxx.gradio.live`.

---
### 📋 Hướng Dẫn Sử Dụng (Chỉ 3 Bước):
1. Chọn menu **Runtime** -> **Change runtime type** -> Đảm bảo đã chọn **T4 GPU** (Miễn phí).
2. Bấm nút **Run All** (hoặc chạy lần lượt 4 ô code bên dưới).
3. Khi ô số 4 hiện dòng `Running on public URL: https://xxxxxxxx.gradio.live`, hãy copy đường link đó và dán vào nút **⚙️ Config -> Colab Gradio Live URL** trên Web Blender Studio!

In [ ]:
# [BƯỚC 1] Kiểm tra cấu hình GPU Colab (Yêu cầu NVIDIA T4 16GB VRAM trở lên)
!nvidia-smi
import torch

print("\n" + "="*60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Sẵn Sàng: {gpu_name}")
    print(f"✅ Dung Lượng VRAM: {vram_gb:.2f} GB (Đáp ứng hoàn hảo cho Hunyuan3D-2.1)")
    print("="*60)
else:
    raise SystemExit("❌ CHƯA BẬT GPU! Hãy vào menu Runtime -> Change runtime type -> Chọn T4 GPU rồi chạy lại.")


In [ ]:
# [BƯỚC 2] Tải mã nguồn Tencent Hunyuan3D-2 và cài đặt thư viện cần thiết
import os, sys

%cd /content
if not os.path.exists("Hunyuan3D-2"):
    print("⏳ Đang clone mã nguồn Tencent Hunyuan3D-2...")
    !git clone --depth 1 https://github.com/Tencent/Hunyuan3D-2.git

%cd /content/Hunyuan3D-2

print("⏳ Cài đặt các thư viện lõi cho DiT & Texture Synthesis...")
# Cài đặt transformers==4.46.0 (chuẩn Tencent) để tương thích 100% trọng số DINOv2
!pip install -q transformers==4.46.0 diffusers==0.30.0 accelerate==1.1.1
!pip install -q ninja pybind11 trimesh pymeshlab pygltflib xatlas rembg onnxruntime einops gradio==5.20.0
!pip install -q -r requirements.txt
!pip install -q -e .

print("✅ Cài đặt thư viện hoàn tất!")


In [ ]:
# [BƯỚC 3] Biên dịch CUDA Custom Rasterizer phục vụ Texture Baking (hy3dpaint)
%cd /content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer
print("⏳ Đang biên dịch CUDA Rasterizer...")
!python3 setup.py install
%cd /content/Hunyuan3D-2

print("✅ CUDA Rasterizer đã được tích hợp thành công!")


In [ ]:
# [BƯỚC 4] Cấu hình Tiết Kiệm RAM (Dưới 8GB) và Khởi Chạy Máy Chủ Hunyuan3D-2.1
# 4.1. Tối ưu hóa tải trọng số Diffusers để không nhân đôi RAM hệ thống (low_cpu_mem_usage=True)
mv_file = 'hy3dgen/texgen/utils/multiview_utils.py'
if os.path.exists(mv_file):
    with open(mv_file, 'r', encoding='utf-8') as f: mvc = f.read()
    mvc = mvc.replace('torch_dtype=torch.float16)', 'torch_dtype=torch.float16, low_cpu_mem_usage=True)')
    with open(mv_file, 'w', encoding='utf-8') as f: f.write(mvc)

de_file = 'hy3dgen/texgen/utils/dehighlight_utils.py'
if os.path.exists(de_file):
    with open(de_file, 'r', encoding='utf-8') as f: dec = f.read()
    dec = dec.replace('safety_checker=None,', 'safety_checker=None, low_cpu_mem_usage=True,')
    with open(de_file, 'w', encoding='utf-8') as f: f.write(dec)

# 4.2. Tự động vá tương thích DINOv2 weights cho mọi phiên bản transformers
pipe_file = "hy3dgen/shapegen/pipelines.py"
if os.path.exists(pipe_file):
    with open(pipe_file, "r", encoding="utf-8") as f:
        pcode = f.read()
    remap_fn = '''def _remap_dinov2_keys(sd, target):
    if any('attention.attention.query' in k for k in target) or not any('attention.attention.query' in k for k in sd):
        return sd
    out = {}
    for k, v in sd.items():
        n = (k.replace('attention.attention.query', 'attention.q_proj').replace('attention.attention.key', 'attention.k_proj')
              .replace('attention.attention.value', 'attention.v_proj').replace('attention.output.dense', 'attention.o_proj'))
        if '.mlp.weights_in.' in n:
            gate, up = v.chunk(2, dim=0)
            out[n.replace('weights_in', 'gate_proj')] = gate; out[n.replace('weights_in', 'up_proj')] = up
            continue
        out[n.replace('.mlp.weights_out.', '.mlp.down_proj.')] = v
    return out
'''
    if "def _remap_dinov2_keys" not in pcode:
        pcode = remap_fn + "\n" + pcode
        pcode = pcode.replace("conditioner.load_state_dict(ckpt['conditioner'])", "conditioner.load_state_dict(_remap_dinov2_keys(ckpt['conditioner'], conditioner.state_dict()))")
        with open(pipe_file, "w", encoding="utf-8") as f:
            f.write(pcode)
        print("✅ Đã kích hoạt DINOv2 Auto-Remap & low_cpu_mem_usage tiết kiệm RAM!")

# 4.3. Cấu hình Gradio Share Tunnel
with open("gradio_app.py", "r", encoding="utf-8") as f:
    code = f.read()

target_pattern = '''    demo = build_app()
    app = gr.mount_gradio_app(app, demo, path="/")
    uvicorn.run(app, host=args.host, port=args.port, workers=1)'''

replace_pattern = '''    demo = build_app()
    print("\n" + "*"*60)
    print("🚀 KHỞI CHẠY GRADIO SHARE TUNNEL CHO WEB BLENDER STUDIO...")
    print("*"*60)
    demo.launch(share=True, server_name="0.0.0.0", server_port=7860, inline=False)'''

if target_pattern in code:
    code = code.replace(target_pattern, replace_pattern)
    with open("gradio_app.py", "w", encoding="utf-8") as f:
        f.write(code)
    print("✅ Đã cấu hình Gradio Tunneling (share=True)!")

print("\n🔥 ĐANG KHỞI ĐỘNG HỆ THỐNG AI HUNYUAN3D-2.1 TRÊN GPU T4 (RAM < 8GB)...")
print("👉 Sau khi máy chủ khởi động, hãy COPY đường link https://xxxxxxxx.gradio.live bên dưới!")

# Khởi chạy với bản Hunyuan3D-2mini tối ưu RAM (chỉ tốn ~1.5GB thay vì 8GB của bản full)
!python3 gradio_app.py \
    --model_path tencent/Hunyuan3D-2mini \
    --subfolder hunyuan3d-dit-v2-mini-turbo \
    --texgen_model_path tencent/Hunyuan3D-2 \
    --low_vram_mode
